In [13]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

True

In [14]:
import urllib
import json

def http(x):
    response = urllib.request.urlopen(x)
    data = response.read()
    return data.decode('utf-8')

def query(x):
    concept = x.lower().replace(' ', '_')
    url = "https://api.conceptnet.io/query?start=/c/en/{}&rel=/r/IsA&limit=10".format(
        urllib.parse.quote(concept))
    try:
        result = json.loads(http(url))
    except Exception:
        return {}
    edges = result.get('edges', [])
    if not edges:
        return {}
    total_weight = sum(edge['weight'] for edge in edges)
    if total_weight == 0:
        return {}
    return {edge['end']['label']: edge['weight'] / total_weight for edge in edges}

query('microsoft')

{}

In [17]:
newsapi_key = os.getenv("API_KEY")
def get_news(country='us'):
    res = json.loads(http("https://newsapi.org/v2/top-headlines?country={0}&apiKey={1}".format(country,newsapi_key)))
    return res['articles']

all_titles = [x['title'] for x in get_news('us')+get_news('gb')]

In [18]:
all_titles

['Live updates: Taylor Swift and Travis Kelce’s multi-day wedding celebration is underway - CNN',
 'California farmer and food marketer spar over who can sell white nectarines - Yahoo',
 'Iran sends defiant message to Trump with colossal funeral for slain Supreme Leader Ali Khamenei - CNN',
 'U.S. warned Iran about Israel’s aims to assassinate leaders - The Washington Post',
 'Madonna Finds a Nonstop Groove on Her Best Album in 20 Years - Rolling Stone',
 'Blake Lively confirms major friendship fallout with Taylor Swift as she skips wedding celebration - Yahoo',
 'Bessent on Trump\'s crypto earnings: "I don\'t think there\'s an appearance problem" - CBS News',
 'Ronaldo scores as Portugal come back to win, Croatia denied by late VAR - Al Jazeera',
 'Sony’s PlayStation disc factory is already being repurposed - The Verge',
 "'Milestone': Scientists claim to build synthetic cell, raising concerns in step toward artificial life - Fox News",
 'Doctor: ‘Don’t panic’ as Michigan tracks cyclo

In [19]:
from textblob import TextBlob

In [24]:
import sys
!{sys.executable} -m textblob.download_corpora

Finished.


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\jsand\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jsand\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jsand\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\jsand\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to
[nltk_data]     C:\Users\jsand\AppData\Roaming\nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\jsand\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is alr

In [25]:
w = {}
for x in all_titles:
    for n in TextBlob(x).noun_phrases:
        if n in w:
            w[n].append(x)
        else:
            w[n]=[x]
{ x:len(w[x]) for x in w.keys()}

{'live': 1,
 'taylor swift': 3,
 'travis kelce': 2,
 '’ s multi-day wedding celebration': 1,
 'cnn': 2,
 'california': 1,
 'food marketer spar': 1,
 'white nectarines': 1,
 'yahoo': 2,
 'iran': 2,
 'sends defiant message': 1,
 'trump': 4,
 'colossal funeral': 1,
 'supreme leader': 1,
 'ali khamenei': 1,
 'u.s.': 1,
 'israel': 1,
 '’ s aims': 1,
 'assassinate leaders': 1,
 'washington post': 2,
 'madonna finds': 1,
 'nonstop groove': 1,
 'album': 1,
 'years': 1,
 'rolling stone': 1,
 'blake lively': 1,
 'major friendship fallout': 1,
 'wedding celebration': 1,
 'bessent': 1,
 "'s crypto earnings": 1,
 'appearance problem': 1,
 'cbs news': 1,
 'ronaldo': 1,
 'portugal': 1,
 'croatia': 1,
 'var': 1,
 'al jazeera': 2,
 'sony': 1,
 '’ s': 1,
 'playstation': 1,
 'disc factory': 1,
 'verge': 1,
 'scientists': 1,
 'synthetic cell': 1,
 'artificial life': 1,
 'fox': 1,
 'don': 1,
 '’ t panic ’': 1,
 'michigan': 1,
 'tracks cyclosporiasis outbreak': 1,
 'woodtv.com': 1,
 'administration indicts'

In [ ]:
w = {}
for x in all_titles:
    for noun in TextBlob(x).noun_phrases:
        terms = query(noun)
        for term in [u for u in terms.keys() if terms[u]>0.1]:
            if term in w:
                w[term].append(x)
            else:
                w[term]=[x]